<a href="https://colab.research.google.com/github/ProductPriceTrackerOrg/data-science/blob/main/notebooks/product-category/01_prepare_the_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import json
import pandas as pd

In [3]:
# Load JSON
with open("/content/2025=08-31.json", "r") as f:
    data_life = json.load(f)

# Load JSON
with open("/content/data.json", "r") as f:
    data_simp = json.load(f)


In [4]:
# Load JSON
with open("/content/appleme_products.json", "r") as f:
    data_appl = json.load(f)


In [5]:
data_appl = data_appl["products"]


In [6]:
# Load JSON
with open("/content/laptop_lk_scrape.json", "r") as f:
    data_lap = json.load(f)
    data_lap = data_lap["products"]

# Load JSON
with open("/content/one1lk_products2025-08-31.json", "r") as f:
    data_one = json.load(f)

# Load JSON
with open("/content/cyberdeals_lk_scrape_optimized.json", "r") as f:
    data_cyb = json.load(f)
    data_cyb = data_cyb["products"]

In [7]:
# concat the all products
products = data_life + data_simp + data_appl + data_lap + data_one + data_cyb

In [8]:
import hashlib

def generate_shop_product_id(pid, shop_name):
    """Generate hex MD5 from product_id_native + shop_name"""
    concat_str = f"{pid}{shop_name}"
    return hashlib.md5(concat_str.encode("utf-8")).hexdigest()

In [9]:
all_records = []

for p in products:
  shop_name = p["metadata"]["source_website"]
  shop_product_id = generate_shop_product_id(p["product_id_native"], shop_name)

  all_records.append({
        "shop_product_id": shop_product_id,
        "shop_name": shop_name,
        "product_title": p["product_title"],
        "category_path": " > ".join(p["category_path"]) if p["category_path"] else None
        })

In [10]:
OUTPUT_FILE = "output/products_summary.csv"

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

df = pd.DataFrame(all_records)

In [11]:
df.shape

(28866, 4)

In [12]:
# print categories whare value count is > 100
df["category_path"].value_counts()[df["category_path"].value_counts() > 20]


,count
category_path,
Electronics > All Products,1073
Tempered Glass,644
Backcovers and Pouches > iPhone Cases,421
Chargers and Adapters,372
Backcovers and Pouches,305
...,...
Brands > Consumer Notebooks > Core i5 Laptops > Gaming Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops,21
Computer Accessories > HDD Enclosures,21
Tablet Screen Protectors,21


In [13]:
# write data print categories whare value count is > 100 to a txt file
with open("output/categories.txt", "w") as f:
    for cat in df["category_path"].value_counts()[df["category_path"].value_counts() > 20].index:
        f.write(f"{cat}\n")


In [14]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

In [19]:
# write data print categories whare value count is > 100 to a txt file
with open("output/categories_2.txt", "w") as f:
    for cat in df["category_path"].value_counts()[(df["category_path"].value_counts() > 10) & (df["category_path"].value_counts() < 20)].index:
        f.write(f"{cat}\n")

In [109]:
import pandas as pd

# Load the CSV we created
df = pd.read_csv("output/products_summary.csv")

# List of exceptions to always keep
exceptions = [
    "Electronics > All Products",
    "Uncategorized",
    "ACCESSORIES",
    "TRAVEL ACCESSORIES",
    "Xiaomi",
    "Brands",
    "Anker",
    "Google"
]

# Count products per category_path
category_counts = df['category_path'].value_counts()

# Categories to keep: count > 10 OR not in exceptions
categories_to_keep = category_counts[category_counts > 10].index.tolist()
categories_to_keep += [c for c in exceptions if c not in categories_to_keep]

# Filter the DataFrame
filtered_df = df[df['category_path'].isin(categories_to_keep)]

In [110]:
# get value count of this category  paths
"""
"Electronics > All Products",
    "Uncategorized",
    "ACCESSORIES",
    "TRAVEL ACCESSORIES",
    "Xiaomi",
    "Brands",
    "Anker",
    "Google"
"""
filtered_df["category_path"].value_counts()

# remove rows where category path is "Electronics > All Products","Uncategorized","ACCESSORIES","TRAVEL ACCESSORIES","Xiaomi","Brands","Anker","Google"
unwanted_categories = [
    "Electronics > All Products",
    "Uncategorized",
    "ACCESSORIES",
    "TRAVEL ACCESSORIES",
    "Xiaomi",
    "Brands",
    "Anker",
    "Google",
    "Computer Peripherals > Computer Peripherals"
]

# Filter out unwanted categories
filtered_df = filtered_df[~filtered_df["category_path"].isin(unwanted_categories)]

# Check the counts again
filtered_df["category_path"].value_counts()



,count
category_path,
Tempered Glass,644
Backcovers and Pouches > iPhone Cases,421
Chargers and Adapters,372
Backcovers and Pouches,305
Mobile Accessories > Chargers and Adapters > Chargers and Adapters,297
...,...
Mobile Phones > Samsung > Samsung,11
Smart Watch Accessories,11
Power Supply,11


In [60]:
filtered_df.shape

(20956, 4)

In [ ]:
import pandas as pd

# Load the CSV
df = pd.read_csv("output/products_summary.csv")

# Categories to remove
exceptions = [
    "Electronics > All Products",
    "Uncategorized",
    "ACCESSORIES",
    "TRAVEL ACCESSORIES",
    "Xiaomi",
    "Brands",
    "Anker",
    "Google"
]

# Step 1: Remove rows where category_path is in exceptions
df = df[~df['category_path'].isin(exceptions)].copy()

In [40]:
df.shape

(27646, 4)

In [41]:
# Step 2: Keep only categories with >= 10 records
category_counts = df['category_path'].value_counts()
categories_to_keep = category_counts[category_counts >= 10].index

filtered_df = df[df['category_path'].isin(categories_to_keep)].copy()

In [42]:
filtered_df.shape

(21416, 4)

In [111]:
category_mapping = {
    # Laptops
    "Electronics > All Products": "Laptops",
    "Brands > Commercial Notebooks > Core i5 Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "HP > Laptops": "Laptops",
    "Brands > Commercial Notebooks > Core i7 Laptops > Gaming Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "Laptops > Apple MacBook > Laptops > Apple MacBook": "Laptops",
    "Laptops > Apple MacBook > Apple MacBook > Laptops": "Laptops",
    "Brands > Commercial Notebooks > Core i3 Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "Laptops > HP > Laptops > HP": "Laptops",
    "Laptops > Microsoft Surface > Laptops > Microsoft Surface": "Laptops",
    "Laptops > HP": "Laptops",
    "Laptops > MSI > Laptops > MSI": "Laptops",
    "Laptops > Asus > Laptops > Asus": "Laptops",
    "Brands > Commercial Notebooks > Core i3 Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "Laptops > Dell > Laptops > Dell": "Laptops",
    "Laptops > HP > HP > Laptops": "Laptops",
    "Laptops > Lenovo": "Laptops",
    "Brands > Consumer Notebooks > Core i5 Laptops > Gaming Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "Brands > Consumer Notebooks > Core i7 Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "Brands > Consumer Notebooks > Core i5 Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "Brands > Commercial Notebooks > Core i7 Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "Laptops > MSI": "Laptops",
    "Laptops > Dell > Dell > Laptops": "Laptops",
    "Laptops": "Laptops",
    "Laptops > Apple MacBook > Laptops > Apple MacBook > On-Demand": "Laptops",
    "Asus > Laptops": "Laptops",
    "Laptops > Lenovo > Laptops > Lenovo": "Laptops",
    "Brands > Consumer Notebooks > Core i9 Laptops > Gaming Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "AMD Ryzen 5 > Brands > Commercial Notebooks > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "HP": "Laptops",
    "Asus": "Laptops",

    # Monitors
    "Monitors": "Monitors",
    "Brands > Monitors > Monitors > Professional Monitors": "Monitors",
    "Computer Peripherals > Monitors > Monitors": "Monitors",
    "Computer Accessories > Monitors": "Monitors",
    "Brands > Gaming Monitors > Monitors > Monitors": "Monitors",
    "Brands > Monitors > Monitors > Professional Monitors > ViewSonic": "Monitors",
    "Computer Peripherals > Monitors > Computer Peripherals > Monitors": "Monitors",
    "Brands > Gaming Monitors > Monitors > Monitors > ViewSonic": "Monitors",
    "Brands > Computer Accessories > Computers & Accessories > Monitors > Monitors > Professional Monitors": "Monitors",
    "Brands > Monitors": "Monitors",
    "Koorui > Monitors": "Monitors",
    "BenQ > Computer Accessories > Monitors": "Monitors",

    # Mobile Phones
    "Mobile Phones > Samsung > Mobile Phones > Samsung": "Mobile Phones",
    "Mobile Phones > Xiaomi > Mobile Phones > Xiaomi": "Mobile Phones",
    "Mobile Phones > Samsung": "Mobile Phones",
    "MOBILE PHONES": "Mobile Phones",
    "Mobile Phones": "Mobile Phones",
    "Mobile Phones > OnePlus > Mobile Phones > OnePlus": "Mobile Phones",
    "Mobile Phones > Nokia > Mobile Phones > Nokia": "Mobile Phones",
    "Mobile Phones > Vivo > Mobile Phones > Vivo": "Mobile Phones",
    "Mobile Phones > Apple > Apple > Mobile Phones": "Mobile Phones",
    "Mobile Phones > Xiaomi": "Mobile Phones",
    "Mobile Phones > Apple > Mobile Phones > Apple": "Mobile Phones",
    "Mobile Phones > Apple": "Mobile Phones",
    "Mobile Phones > Nokia": "Mobile Phones",
    "Mobile Phones > Oppo > Mobile Phones > Oppo": "Mobile Phones",
    "Mobile Phones > ZTE > Mobile Phones > ZTE": "Mobile Phones",
    "Mobile Phones > Realme > Mobile Phones > Realme": "Mobile Phones",
    "Mobile Phones > Redmi": "Mobile Phones",
    "Mobile Phones > Honor > Honor > Mobile Phones": "Mobile Phones",
    "Mobile Phones > Infinix > Mobile Phones > Infinix": "Mobile Phones",
    "Mobile Phones > UMIDIGI > Mobile Phones > UMIDIGI": "Mobile Phones",
    "Mobile Phones > Google > Mobile Phones > Google": "Mobile Phones",
    "Mobile Phones > HTC": "Mobile Phones",
    "Mobile Phones > ZTE": "Mobile Phones",
    "Mobile Phones > OnePlus": "Mobile Phones",
    "Mobile Phones > Nothing Phone (1) > Mobile Phones > Nothing Phone (1)": "Mobile Phones",
    "Mobile Phones > Huawei": "Mobile Phones",
    "Mobile Phones > Huawei > Mobile Phones > Huawei": "Mobile Phones",
    "Mobile Phones > Google > Google > Mobile Phones": "Mobile Phones",
    "Mobile Phones > Huawei > Huawei > Mobile Phones": "Mobile Phones",
    "Mobile Phones > Oppo": "Mobile Phones",
    "HONOR > Mobile Phones": "Mobile Phones",
    "Mobile Phones > Unit Only": "Mobile Phones",
    "Mobile Phones > Tecno": "Mobile Phones",
    "Infinix > Mobile Phones": "Mobile Phones",
    "Mobile Phones > BlackBerry": "Mobile Phones",
    "ZTE": "Mobile Phones",
    "Xiaomi Redmi": "Mobile Phones",
    "Samsung": "Mobile Phones",
    "Google": "Mobile Phones",
    "Honor": "Mobile Phones",
    "Xiaomi": "Mobile Phones",
    "Infinix": "Mobile Phones",
    "Mobile Phones > HOTWAV > HOTWAV > Mobile Phones": "Mobile Phones",
    "Mobile Phones > TCL > Mobile Phones > TCL": "Mobile Phones",
    "Mobile Phones > Nokia > Nokia": "Mobile Phones",
    "Mobile Phones > Samsung > Samsung": "Mobile Phones",

    # Tablets
    "Tablets > Samsung > Tablets > Samsung": "Tablets",
    "Tablets > iPad > iPad > Tablets": "Tablets",
    "Tablets > iPad > Tablets > iPad": "Tablets",
    "Tablets > Samsung > Samsung > Tablets": "Tablets",
    "Tablets > Xiaomi > Tablets > Xiaomi": "Tablets",
    "Amazon > Tablets": "Tablets",
    "Tablets > Amazon > Tablets > Amazon": "Tablets",
    "Apple iPad > Tablets": "Tablets",
    "Tablets > Amazon > Amazon > Tablets": "Tablets",
    "Samsung > Tablets": "Tablets",
    "Tablets": "Tablets",
    "Samsung Tabs": "Tablets",

    # Smart Watches & Accessories
    "SMART WATCHES": "Smart Watches & Accessories",
    "Smart Watches > Apple Watch > Apple Watch > Smart Watches": "Smart Watches & Accessories",
    "Smart Watches > Apple Watch > Smart Watches > Apple Watch": "Smart Watches & Accessories",
    "Apple Watch > Smart Watches": "Smart Watches & Accessories",
    "Smart Watches > Xiaomi > Smart Watches > Xiaomi": "Smart Watches & Accessories",
    "Smart Watches > Xiaomi": "Smart Watches & Accessories",
    "Samsung > Smart Watches": "Smart Watches & Accessories",
    "Smart Watches > Amazfit > Smart Watches > Amazfit": "Smart Watches & Accessories",
    "Smart Watches > Telzeal": "Smart Watches & Accessories",
    "Haino Teko > Smart Watches": "Smart Watches & Accessories",
    "Smart Watches": "Smart Watches & Accessories",
    "Smart Watches > Samsung > Smart Watches > Samsung": "Smart Watches & Accessories",
    "Green Lion > Smart Watches": "Smart Watches & Accessories",
    "Apple Watch Straps": "Smart Watches & Accessories",
    "Apple Watch Cases": "Smart Watches & Accessories",
    "Smart Watch Accessories > Smart Watch Accessories": "Smart Watches & Accessories",
    "Apple Watch Tempered Glass": "Smart Watches & Accessories",
    "Watch Straps and Bands": "Smart Watches & Accessories",
    "Smart Watches > Smart Watches": "Smart Watches & Accessories",
    "Fitness Trackers > Xiaomi > Fitness Trackers > Xiaomi": "Smart Watches & Accessories",
    "Mobile Accessories > Apple Accessories > Apple Watch Straps": "Smart Watches & Accessories",
    "Apple Accessories > Apple Watch Straps": "Smart Watches & Accessories",
    "Mobile Accessories > Apple Accessories > Apple Watch Case > Apple Accessories > Apple Watch Case": "Smart Watches & Accessories",
    "Smart Watch Accessories": "Smart Watches & Accessories",
    "Fitness Trackers > Xiaomi": "Smart Watches & Accessories",
    "Amazfit > Smart Watches": "Smart Watches & Accessories",
    "SMART WATCHES > KIESLECT": "Smart Watches & Accessories",
    "Smart Watches > Huawei > Huawei > Smart Watches": "Smart Watches & Accessories",
    "Huawei > Smart Watches": "Smart Watches & Accessories",
    "Fitbit > Fitness Trackers": "Smart Watches & Accessories",
    "Mobile Accessories > Apple Accessories > Apple Watch Tempered Glass": "Smart Watches & Accessories",

    # Keyboards
    "Keyboards": "Keyboards",
    "Computer Peripherals > Keyboard > Keyboard": "Keyboards",
    "Computer Peripherals > Keyboard > Computer Peripherals > Keyboard": "Keyboards",
    "Computer Accessories > Keyboards": "Keyboards",
    "Gaming Keyboards > Gaming Peripherals": "Keyboards",
    "Gaming Keyboards": "Keyboards",
    "Keyboards > Mouse": "Keyboards",
    "Armaggeddon > Gaming Keyboards > Gaming Peripherals > Keyboards": "Keyboards",
    "Gaming Keyboards > Gaming Peripherals > Keyboards": "Keyboards",
    "Keyboards > WiWU": "Keyboards",
    "Mobile Accessories > Wireless Keyboards > Wireless Keyboards": "Keyboards",
    "iPad and Tablet Accessories > Smart Keyboard Folio": "Keyboards",
    "iPad and Tablet Accessories > Smart Keyboard Folio > iPad and Tablet Accessories > Smart Keyboard Folio": "Keyboards",
    "Keyboard Protectors > Laptop Accessories": "Keyboards",

    # Mice
    "COMPUTER ACCESSORIES > MOUSE": "Mice",
    "Logitech > Mouse": "Mice",
    "Computer Peripherals > Mouse > Logitech > Mouse > Logitech": "Mice",
    "Computer Accessories > Gaming Mouse > Gaming Peripherals": "Mice",
    "Gaming Mouse > Gaming Peripherals": "Mice",
    "Computer Accessories > Mouse": "Mice",
    "Computer Peripherals > Mouse > Logitech > Logitech > Mouse": "Mice",
    "Accessories > Brands > Logitech > Wireless Mouse": "Mice",
    "Gaming Mouse > Mouse": "Mice",
    "Gaming Mouse": "Mice",
    "Mouse": "Mice",
    "Alcatroz > Mouse": "Mice",
    "Havit > Mouse": "Mice",
    "Computer Peripherals > Mouse > Xiaomi > Computer Peripherals > Mouse > Xiaomi": "Mice",
    "Gaming Mouse Pads > Gaming Peripherals > Mouse Pads": "Mice",
    "Gaming Mouse Pads": "Mice",
    "Mouse Pads": "Mice",

    # Headphones & Earbuds
    "Headphones & Earbuds": "Headphones & Earbuds",
    "Earphones": "Headphones & Earbuds",
    "Bluetooth Earbuds > Xiaomi": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > Xiaomi > Bluetooth Earbuds > Xiaomi": "Headphones & Earbuds",
    "Mobile Accessories > Headphones > JBL > Headphones > JBL": "Headphones & Earbuds",
    "Bluetooth Earbuds > Lenovo": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > Lenovo > Bluetooth Earbuds > Lenovo": "Headphones & Earbuds",
    "Bluetooth Earbuds > JBL": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > JBL > Bluetooth Earbuds > JBL": "Headphones & Earbuds",
    "Bluetooth Earbuds > Joyroom": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > Joyroom > Bluetooth Earbuds > Joyroom": "Headphones & Earbuds",
    "Celebrat > Earphones": "Headphones & Earbuds",
    "Mobile Accessories > Earphones > JBL > Earphones > JBL": "Headphones & Earbuds",
    "Mobile Accessories > Earphones > Remax > Earphones > Remax": "Headphones & Earbuds",
    "Bluetooth Earbuds > SOUNDPEATS": "Headphones & Earbuds",
    "Earphones > Joyroom": "Headphones & Earbuds",
    "Mobile Accessories > Earphones > Joyroom > Earphones > Joyroom": "Headphones & Earbuds",
    "Earphones": "Headphones & Earbuds",
    "Bluetooth Earbuds": "Headphones & Earbuds",
    "Anker > Bluetooth Earbuds": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > Anker > Bluetooth Earbuds > Anker": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > Anker > Anker > Bluetooth Earbuds": "Headphones & Earbuds",
    "Baseus > Bluetooth Earbuds": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > OnePlus > Bluetooth Earbuds > OnePlus": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > Bluetooth Earbuds": "Headphones & Earbuds",
    "Earbuds Accessories": "Headphones & Earbuds",
    "Mobile Accessories > Headphones > Jabra > Headphones > Jabra": "Headphones & Earbuds",
    "Celebrat > Headphones": "Headphones & Earbuds",
    "Headphones": "Headphones & Earbuds",
    "Headphones > Sony": "Headphones & Earbuds",
    "Headphones > Remax": "Headphones & Earbuds",
    "Anker > Headphones": "Headphones & Earbuds",
    "Bluetooth Earbuds > Sony": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > Sony > Bluetooth Earbuds > Sony": "Headphones & Earbuds",
    "Bluetooth Earbuds > Remax": "Headphones & Earbuds",
    "Mobile Accessories > Headphones > Sony > Headphones > Sony": "Headphones & Earbuds",
    "Mobile Accessories > Headphones > Anker > Headphones > Anker": "Headphones & Earbuds",
    "Mobile Accessories > Headphones > Skullcandy > Headphones > Skullcandy": "Headphones & Earbuds",
    "Mobile Accessories > Bluetooth Earbuds > Skullcandy > Bluetooth Earbuds > Skullcandy": "Headphones & Earbuds",
    "Baseus > Earphones": "Headphones & Earbuds",
    "Mobile Accessories > Earphones > Anker > Earphones > Anker": "Headphones & Earbuds",
    "Mobile Accessories > Earphones > Lenovo > Earphones > Lenovo": "Headphones & Earbuds",
    "Earphones > JBL": "Headphones & Earbuds",
    "Mobile Accessories > Earphones > Xiaomi > Earphones > Xiaomi": "Headphones & Earbuds",
    "Earphones > WK": "Headphones & Earbuds",
    "Earphones > Xiaomi": "Headphones & Earbuds",
    "Mobile Accessories > Earphones > Earphones": "Headphones & Earbuds",
    "Gaming Headphones": "Headphones & Earbuds",
    "Gaming Headphones > Gaming Peripherals": "Headphones & Earbuds",
    "Headsets": "Headphones & Earbuds",
    "Headphones > JBL": "Headphones & Earbuds",
    "Earphones > Remax": "Headphones & Earbuds",

    # Speakers
    "AUDIO SPEAKERS > BLUETHOOTH SPEAKERS": "Speakers",
    "Bluetooth Speakers": "Speakers",
    "Anker > Bluetooth Speakers": "Speakers",
    "Bluetooth Speakers > Remax": "Speakers",
    "Bluetooth Speakers > JBL": "Speakers",
    "Audio Speakers and Subwoofers": "Speakers",
    "Bluetooth Speakers > Bose > Bluetooth Speakers > Bose": "Speakers",
    "Bluetooth Speakers > Bluetooth Speakers": "Speakers",
    "AUDIO SPEAKERS > BLUETHOOTH SPEAKERS > JBL": "Speakers",
    "Anker > Bluetooth Speakers": "Speakers",
    "Bluetooth Speakers > Microlab > PC Speakers": "Speakers",
    "Bluetooth Speakers > JBL > Bluetooth Speakers > JBL": "Speakers",
    "Bluetooth Speakers > Celebrat": "Speakers",
    "Bluetooth Speakers > Remax > Bluetooth Speakers > Remax": "Speakers",
    "Audio Speakers and Subwoofers > JBL > Audio Speakers and Subwoofers > JBL": "Speakers",
    "Computer Accessories > PC Speakers": "Speakers",
    "PC Speakers": "Speakers",
    "Bluetooth Speakers > Anker > Bluetooth Speakers > Anker": "Speakers",
    "Bluetooth Speakers > Anker > Anker > Bluetooth Speakers": "Speakers",
    "Bluetooth Speakers > Tronsmart": "Speakers",
    "Bluetooth Speakers > Monster": "Speakers",
    "Bluetooth Speakers > Harman Kardon": "Speakers",
    "Bluetooth Speakers > Harman Kardon > Bluetooth Speakers > Harman Kardon": "Speakers",
    "Bluetooth Speakers > Marshall > Bluetooth Speakers > Marshall": "Speakers",
    "Bluetooth Speakers > Marshall": "Speakers",
    "Bluetooth Speakers > Xiaomi": "Speakers",
    "Bluetooth Speakers > Outdoor Speakers > Remax > Outdoor Speakers > Remax": "Speakers",
    "Amazon > Smart Speakers": "Speakers",
    "AUDIO SPEAKERS > BLUETHOOTH SPEAKERS > ANKER": "Speakers",

    # Webcams & Microphones
    "Web Camera": "Webcams & Microphones",
    "Microphones": "Webcams & Microphones",
    "Mobile Accessories > Microphones > Microphones": "Webcams & Microphones",
    "Computer Peripherals > Web Camera > Web Camera": "Webcams & Microphones",
    "Microphones > Mobile Accessories": "Webcams & Microphones",

    # Chargers & Power Banks
    "Chargers and Adapters": "Chargers & Power Banks",
    "Mobile Accessories > Chargers and Adapters > Chargers and Adapters": "Chargers & Power Banks",
    "PHONE ACCESSORIES > CHRGERS AND ADEPTORS": "Chargers & Power Banks",
    "Car Accessories > Chargers and Adapters": "Chargers & Power Banks",
    "PHONE ACCESSORIES > WIRELESS CHARGERS": "Chargers & Power Banks",
    "Chargers and Adapters > Mobile Accessories": "Chargers & Power Banks",
    "Mobile Accessories > Chargers and Adapters > Mobile Accessories > Chargers and Adapters": "Chargers & Power Banks",
    "Car Accessories > Chargers and Adapters > Car Accessories > Chargers and Adapters": "Chargers & Power Banks",
    "Laptop Accessories > Power Adapters and Chargers > Laptop Accessories > Power Adapters and Chargers": "Chargers & Power Banks",
    "Power Banks": "Chargers & Power Banks",
    "Baseus > Power Banks": "Chargers & Power Banks",
    "Power Banks > Remax": "Chargers & Power Banks",
    "Mobile Accessories > Power Banks > Remax > Power Banks > Remax": "Chargers & Power Banks",
    "Power Banks > Xiaomi": "Chargers & Power Banks",
    "Power Banks > UGREEN": "Chargers & Power Banks",
    "Anker > Power Banks": "Chargers & Power Banks",
    "Mobile Accessories > Power Banks > Anker > Power Banks > Anker": "Chargers & Power Banks",
    "Aspor > Power Banks": "Chargers & Power Banks",
    "Mobile Accessories > Power Banks > Baseus > Power Banks > Baseus": "Chargers & Power Banks",
    "Mobile Accessories > Power Banks > Power Banks": "Chargers & Power Banks",
    "Mobile Accessories > Power Banks > Baseus > Baseus > Power Banks": "Chargers & Power Banks",
    "Mobile Accessories > Power Banks > Mi > Mi > Power Banks": "Chargers & Power Banks",
    "Mobile Accessories > Power Banks > Anker > Anker": "Chargers & Power Banks",
    "Mobile Accessories > Power Banks > Anker > Anker > Power Banks": "Chargers & Power Banks",
    "Wireless Chargers": "Chargers & Power Banks",
    "Mobile Accessories > Wireless Chargers > WiWU > Wireless Chargers > WiWU": "Chargers & Power Banks",
    "Mobile Accessories > Wireless Chargers > Wireless Chargers": "Chargers & Power Banks",
    "Baseus > Wireless Chargers": "Chargers & Power Banks",
    "Mobile Accessories > Chargers and Adapters": "Chargers & Power Banks",
    "Asus Chargers & Adapters > Laptop Chargers and Adapters": "Chargers & Power Banks",
    "Apple MacBook Chargers & Adapters": "Chargers & Power Banks",
    "Laptop Chargers and Adapters > Lenovo Chargers & Adapters": "Chargers & Power Banks",
    "Computer Accessories > Power Supply": "Chargers & Power Banks",
    "Brands > Components > Corsair > Power Supply": "Chargers & Power Banks",
    "Power Supply": "Chargers & Power Banks",

    # Cables & Adapters
    "PHONE ACCESSORIES > CABLES & DOCKS": "Cables & Adapters",
    "Computer Peripherals > Hubs and Adapters > Hubs and Adapters": "Cables & Adapters",
    "USB-C Cables": "Cables & Adapters",
    "Hubs and Adapters": "Cables & Adapters",
    "Lightning Cables": "Cables & Adapters",
    "Computer Accessories": "Cables & Adapters",
    "Home Accessories > Cables and Connectors > Cables and Connectors": "Cables & Adapters",
    "HDMI Cables": "Cables & Adapters",
    "Computer Peripherals > Hubs and Adapters > Computer Peripherals > Hubs and Adapters": "Cables & Adapters",
    "Computer Accessories > Hubs and Adapters": "Cables & Adapters",
    "Mobile Accessories > USB Type-C Cables > USB Type-C Cables": "Cables & Adapters",
    "Mobile Accessories > USB Type-C Cables": "Cables & Adapters",
    "Mobile Accessories > Apple Accessories > Lightning Cable > Apple Accessories > Lightning Cable": "Cables & Adapters",
    "Apple Accessories > Lightning Cables": "Cables & Adapters",
    "Mobile Accessories > Apple Accessories > Lightning Cable > Lightning Cable": "Cables & Adapters",
    "Lightning Cables > Micro USB Cables > USB-C Cables": "Cables & Adapters",
    "Mobile Accessories > Apple Accessories > Lightning Cable": "Cables & Adapters",
    "Micro USB Cables": "Cables & Adapters",
    "HDMI Cables > VGA Cables": "Cables & Adapters",
    "Hubs and Adapters > Ugreen": "Cables & Adapters",
    "Hubs and Adapters > WiWU": "Cables & Adapters",
    "Baseus > Hubs and Adapters": "Cables & Adapters",
    "PHONE ACCESSORIES > DOCKS": "Cables & Adapters",
    "Lightning Adapters": "Cables & Adapters",
    "Mobile Adapters": "Cables & Adapters",
    "Mobile Accessories > Mobile Adapters > Mobile Adapters": "Cables & Adapters",
    "Mobile Accessories > Mobile Adapters": "Cables & Adapters",
    "PHONE ACCESSORIES > MOBILE ADAPTERS": "Cables & Adapters",
    "USB WiFi Adapters": "Cables & Adapters",
    "Computer Accessories > USB WiFi Adapters": "Cables & Adapters",
    "Cables and Connectors": "Cables & Adapters",
    "Home Accessories > Cables and Connectors > HDMI Cable > Cables and Connectors > HDMI Cable": "Cables & Adapters",
    "Cables and Connectors > HDMI Cables": "Cables & Adapters",
    "Mobile Accessories > AUX Cables > AUX Cables": "Cables & Adapters",
    "AUX Cables": "Cables & Adapters",
    "Baseus > USB-C Cables": "Cables & Adapters",
    "Anker > USB-C Cables": "Cables & Adapters",
    "USB Type-C Cables": "Cables & Adapters",
    "Accessories > Brands > Cables > HDMI Cables > Ugreen > Ugreen > Video Cables & Video Extender": "Cables & Adapters",
    "Docking Stations": "Cables & Adapters",
    "LAN Accessories": "Cables & Adapters",
    "Cables > Other Accessories": "Cables & Adapters",
    "Adapter & Hub > Other Accessories": "Cables & Adapters",
    "Extension Power Cords": "Cables & Adapters",
    "Home Accessories > Extension Power Cords > Extension Power Cords": "Cables & Adapters",
    "Extension Power Cords > Power & Cables": "Cables & Adapters",
    "Mobile Accessories > USB Type-C Cables > Chargers and Adapters > USB Type-C Cables": "Cables & Adapters",

    # Cases & Screen Protectors
    "Tempered Glass": "Cases & Screen Protectors",
    "Backcovers and Pouches > iPhone Cases": "Cases & Screen Protectors",
    "Backcovers and Pouches": "Cases & Screen Protectors",
    "Apple Accessories > iPhone Cases": "Cases & Screen Protectors",
    "Mobile Accessories > Tempered Glass": "Cases & Screen Protectors",
    "Mobile Accessories > Backcovers and Pouches > Backcovers and Pouches": "Cases & Screen Protectors",
    "iPhone Cases": "Cases & Screen Protectors",
    "iPhone Screen Protectors": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPhone Tempered Glass > Apple iPhone Tempered Glass": "Cases & Screen Protectors",
    "PHONE ACCESSORIES > PHONE CASES": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > Tablet Screen Protectors": "Cases & Screen Protectors",
    "Mobile Accessories > Tempered Glass > Tempered Glass": "Cases & Screen Protectors",
    "PHONE ACCESSORIES > TEMPERED GLASS": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPhone Case > Apple Accessories > Apple iPhone Case": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPhone Case > Apple iPhone Case": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > Tempered Glass > iPad and Tablet Accessories > Tempered Glass": "Cases & Screen Protectors",
    "iPhone Screen Protectors > Tempered Glass": "Cases & Screen Protectors",
    "iPhone Camera Lens Protectors": "Cases & Screen Protectors",
    "Mobile Accessories > Backcovers and Pouches": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > Cases and Pouches > iPad and Tablet Accessories > Cases and Pouches": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > iPad Cases > Smart Cases and Covers": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPhone Case": "Cases & Screen Protectors",
    "Backcovers and Pouches > Mobile Accessories": "Cases & Screen Protectors",
    "Laptop Hard Shell Case": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > Tempered Glass": "Cases & Screen Protectors",
    "Laptop Accessories > Screen Protectors": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPhone Case > Mobile Accessories > Apple Accessories > Apple iPhone Case": "Cases & Screen Protectors",
    "Cases and Pouches > iPad and Tablet Accessories": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > iPad and Tablet Accessories": "Cases & Screen Protectors",
    "Mobile Accessories > Camera Tempered Glass": "Cases & Screen Protectors",
    "Apple AirPods Cases": "Cases & Screen Protectors",
    "Mobile Accessories > Backcovers and Pouches > Mobile Accessories > Backcovers and Pouches": "Cases & Screen Protectors",
    "Mobile Accessories > Phone Cases": "Cases & Screen Protectors",
    "iPad Cases > Smart Cases and Covers": "Cases & Screen Protectors",
    "Apple Accessories > iPhone Screen Protectors": "Cases & Screen Protectors",
    "Apple Accessories > Apple AirPods Cases": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > Smart Cases and Covers": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > Smart Cases and Covers > iPad and Tablet Accessories > Smart Cases and Covers": "Cases & Screen Protectors",
    "Cases and Pouches > iPad and Tablet Accessories > Smart Cases and Covers": "Cases & Screen Protectors",
    "iPad and Tablet Accessories > Cases and Pouches > Cases and Pouches > iPad and Tablet Accessories": "Cases & Screen Protectors",
    "Apple Accessories > iPad and Tablet Accessories > iPad Cases > Smart Cases and Covers": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPad Case > Apple iPad Case": "Cases & Screen Protectors",
    "Tablet Screen Protectors": "Cases & Screen Protectors",
    "Camera Tempered Glass": "Cases & Screen Protectors",
    "PHONE ACCESSORIES > iPHONE CASES": "Cases & Screen Protectors",
    "PHONE ACCESSORIES > APPLE PHONE CASES": "Cases & Screen Protectors",
    "PHONE ACCESSORIES > CASES & SCREEN PROTECTORS": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPhone Tempered Glass": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPhone Tempered Glass > Apple Accessories > Apple iPhone Tempered Glass": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple iPhone Tempered Glass > Apple iPhone Tempered Glass > Tempered Glass": "Cases & Screen Protectors",

    # Bags, Sleeves & Backpacks
    "Laptop Bags": "Bags, Sleeves & Backpacks",
    "Backpacks and Bags > Laptop Bags > Laptop Bags": "Bags, Sleeves & Backpacks",
    "Backpacks and Bags > Laptop Sleeves > Backpacks and Bags > Laptop Sleeves": "Bags, Sleeves & Backpacks",
    "Backpacks and Bags > Backpacks > Backpacks and Bags > Backpacks": "Bags, Sleeves & Backpacks",
    "Backpacks and Bags > Laptop Bags > Backpacks and Bags > Laptop Bags": "Bags, Sleeves & Backpacks",
    "Backpacks and Bags > Storage Bags": "Bags, Sleeves & Backpacks",
    "Backpacks > Backpacks and Bags": "Bags, Sleeves & Backpacks",
    "Backpacks and Bags > Laptop Sleeves": "Bags, Sleeves & Backpacks",
    "Backpacks and Bags > Backpacks > Backpacks > Backpacks and Bags": "Bags, Sleeves & Backpacks",
    "Laptop Accessories > Laptop Accessories": "Bags, Sleeves & Backpacks",

    # CPUs (Processors)
    "Computer Accessories > Processor": "CPUs (Processors)",

    # Motherboards
    "Motherboards > MSI": "Motherboards",
    "Brands > Components > Motherboard": "Motherboards",

    # RAM (Memory)
    "RAM": "RAM (Memory)",
    "Laptop Accessories > Laptop RAM": "RAM (Memory)",
    "Corsair > RAM": "RAM (Memory)",

    # Storage (SSD, HDD, Pen Drives)
    "Computer Accessories > SSD Drives": "Storage (SSD, HDD, Pen Drives)",
    "SSD Drives": "Storage (SSD, HDD, Pen Drives)",
    "Kingston > SSD Drives": "Storage (SSD, HDD, Pen Drives)",
    "Computer Peripherals > External Hard Drives > Computer Peripherals > External Hard Drives": "Storage (SSD, HDD, Pen Drives)",
    "Computer Peripherals > Flash Drives & Pen Drives > Flash Drives & Pen Drives > SanDisk": "Storage (SSD, HDD, Pen Drives)",
    "Flash Drives & Pen Drives": "Storage (SSD, HDD, Pen Drives)",
    "Computer Accessories > External Hard Drives": "Storage (SSD, HDD, Pen Drives)",
    "Computer Peripherals > Flash Drives & Pen Drives > Flash Drives & Pen Drives > Kingston": "Storage (SSD, HDD, Pen Drives)",
    "Flash Drives & Pen Drives > SanDisk": "Storage (SSD, HDD, Pen Drives)",
    "Memory Cards > SanDisk": "Storage (SSD, HDD, Pen Drives)",
    "PHONE ACCESSORIES > SD CARD & PEN DRIVES": "Storage (SSD, HDD, Pen Drives)",
    "Memory Cards": "Storage (SSD, HDD, Pen Drives)",
    "Mobile Accessories > Memory Cards > SanDisk > Memory Cards > SanDisk": "Storage (SSD, HDD, Pen Drives)",
    "Computer Accessories > Portable Drives": "Storage (SSD, HDD, Pen Drives)",
    "Brands > Components > Computer Accessories > Computers & Accessories > SSD > SSD": "Storage (SSD, HDD, Pen Drives)",
    "Computer Accessories > HDD Enclosures": "Storage (SSD, HDD, Pen Drives)",
    "Computer Accessories > Memory Card Reader": "Storage (SSD, HDD, Pen Drives)",
    "ADATA > SSD Drives": "Storage (SSD, HDD, Pen Drives)",
    "ADATA": "Storage (SSD, HDD, Pen Drives)",

    # Graphic Cards (VGA)
    "Graphic Cards": "Graphic Cards (VGA)",
    "Computer Accessories > Graphic Cards": "Graphic Cards (VGA)",
    "Brands > Components > VGA Cards": "Graphic Cards (VGA)",

    # Power Supplies & PC Cooling
    "Computer Accessories > PC Cooling & Fans": "Power Supplies & PC Cooling",
    "PC Cooling & Fans": "Power Supplies & PC Cooling",
    "Brands > Components > Cooling > NZXT": "Power Supplies & PC Cooling",

    # Networking (Routers)
    "Computer Peripherals > WiFi Routers > WiFi Routers": "Networking (Routers)",
    "WiFi Routers": "Networking (Routers)",
    "Computer Peripherals > WiFi Routers > Computer Peripherals > WiFi Routers": "Networking (Routers)",
    "Computer Accessories > WiFi Routers": "Networking (Routers)",
    "Computer Peripherals > WiFi Extenders and Repeaters > WiFi Extenders and Repeaters": "Networking (Routers)",
    "Network Hubs and Switches > Reyee by Ruijie": "Networking (Routers)",

    # Printers & Scanners
    "Printers > Printer Accessories > Printer Accessories": "Printers & Scanners",
    "Printers > Printers": "Printers & Scanners",
    "HP > Printers": "Printers & Scanners",
    "Projectors": "Printers & Scanners",

    # Gaming Peripherals (Chairs, etc.)
    "Gaming Peripherals > Joysticks and Controllers": "Gaming Peripherals (Chairs, etc.)",
    "Computer Peripherals": "Gaming Peripherals (Chairs, etc.)",
    "Gaming Chairs": "Gaming Peripherals (Chairs, etc.)",
    "PS4 Games": "Gaming Peripherals (Chairs, etc.)",
    "Xbox One Games": "Gaming Peripherals (Chairs, etc.)",
    "PS4 Games > Video Games": "Gaming Peripherals (Chairs, etc.)",

    # Cameras & Drones
    "Cameras > Security Cameras": "Cameras & Drones",
    "Security Cameras": "Cameras & Drones",
    "Smart Home Devices > Security Cameras > Xiaomi > Security Cameras > Xiaomi": "Cameras & Drones",
    "Cameras > Action Cameras > DJI > Action Cameras > DJI": "Cameras & Drones",
    "Cameras > Action Cameras > Action Cameras": "Cameras & Drones",
    "Cameras": "Cameras & Drones",
    "Car Accessories > Dashboard Cameras > Car Accessories > Dashboard Cameras": "Cameras & Drones",
    "Car Accessories > Dashboard Cameras": "Cameras & Drones",

    # Camera Accessories (Gimbals, Tripods)
    "Tripods": "Camera Accessories (Gimbals, Tripods)",
    "Selfie Stick & Monopods": "Camera Accessories (Gimbals, Tripods)",
    "Mobile Accessories > Tripods > Tripods": "Camera Accessories (Gimbals, Tripods)",
    "Selfie Stick & Monopods > Tripods": "Camera Accessories (Gimbals, Tripods)",
    "Camera Accessories > Gimbals": "Camera Accessories (Gimbals, Tripods)",
    "Cameras > Camera Accessories > Gimbals > Camera Accessories > Gimbals": "Camera Accessories (Gimbals, Tripods)",
    "Mobile Accessories > Apple Accessories > Apple iPhone Camera Lens > Apple Accessories > Apple iPhone Camera Lens": "Camera Accessories (Gimbals, Tripods)",
    "Mobile Accessories > Apple Accessories > Apple iPhone Camera Lens": "Camera Accessories (Gimbals, Tripods)",
    "PHONE ACCESSORIES > iPHONE CAMERA LENS": "Camera Accessories (Gimbals, Tripods)",
    "Gimbals": "Camera Accessories (Gimbals, Tripods)",
    "Laptop Accessories > Wireless Presenter > Wireless Presenter": "Camera Accessories (Gimbals, Tripods)",

    # Car Accessories
    "Car Accessories > Phone Holders": "Car Accessories",
    "Car Accessories": "Car Accessories",
    "Car Accessories > Car Accessories": "Car Accessories",
    "Car Accessories > Phone Holders > Car Accessories > Phone Holders": "Car Accessories",
    "Car Accessories > Phone Holders > Phone Holders": "Car Accessories",
    "Phone Holders": "Car Accessories",
    "PHONE ACCESSORIES > PHONE HOLDERS": "Car Accessories",
    "Mobile Stand": "Car Accessories",
    "Audio Transmitters > Car Accessories": "Car Accessories",
    "Car Accessories > Other Accessories": "Car Accessories",

    # Smart Home & Office Accessories
    "Home Accessories > Home Accessories": "Smart Home & Office Accessories",
    "Home Accessories": "Smart Home & Office Accessories",
    "Smart Home Devices > Smart Home Devices": "Smart Home & Office Accessories",
    "Smart Home Devices": "Smart Home & Office Accessories",
    "Streaming Devices > TVs & Accessories": "Smart Home & Office Accessories",
    "TVs > Streaming Devices > Streaming Devices": "Smart Home & Office Accessories",
    "Streaming Devices": "Smart Home & Office Accessories",
    "TVs > Streaming Devices > TVs > Streaming Devices": "Smart Home & Office Accessories",
    "Emergency Lights": "Smart Home & Office Accessories",
    "Laptop Stand": "Smart Home & Office Accessories",
    "Laptop Accessories > Laptop Stand": "Smart Home & Office Accessories",

    # Health & Personal Care Electronics
    "Clippers & Trimmers > Health & Personal Care": "Health & Personal Care Electronics",
    "Health & Personal Care > Health & Personal Care": "Health & Personal Care Electronics",
    "Health & Personal Care > Hair Trimmer > Health & Personal Care > Hair Trimmer": "Health & Personal Care Electronics",
    "Health & Personal Care > Massager > Massager > Health & Personal Care": "Health & Personal Care Electronics",
    "Health & Personal Care": "Health & Personal Care Electronics",
    "Health & Personal Care > Massager": "Health & Personal Care Electronics",
    "Hair Straightener > Health & Personal Care": "Health & Personal Care Electronics",
    "Hair Dryer > Health & Personal Care": "Health & Personal Care Electronics",
    "Dyson > Hair care": "Health & Personal Care Electronics",

    # Tablets (additional mappings)
    "iPad and Tablet Accessories > Stylus and Capacitive Pens": "Tablets",
    "Mobile Accessories > Capacitive Pens > Capacitive Pens": "Tablets",
    "iPad and Tablet Accessories > Capacitve and Stylus Pens > iPad and Tablet Accessories > Capacitve and Stylus Pens": "Tablets",
    "Stylus and Capacitive Pens": "Tablets",

    # Mobile Phone Accessories that should map to Cases & Screen Protectors
    "PHONE ACCESSORIES": "Cases & Screen Protectors",
    "Mobile Accessories > Mobile Accessories": "Cases & Screen Protectors",
    "Mobile Accessories > Apple Accessories > Apple Accessories": "Cases & Screen Protectors",

    # Battery related items - map to Chargers & Power Banks
    "PHONE ACCESSORIES > BATTERY": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Samsung": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Samsung > For Samsung": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Nokia": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For LG": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Apple": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Apple > For Apple": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Huawei > For Huawei": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Micromax": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Nokia > For Nokia": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Micromax > For Micromax": "Chargers & Power Banks",
    "Mobile Accessories > Replacement Batteries > For Huawei": "Chargers & Power Banks",

    # Brands and Uncategorized - map to most likely category
    "Brands": "Laptops",
    "Uncategorized > Uncategorized": "Smart Home & Office Accessories",
    "Uncategorized": "Smart Home & Office Accessories",
    "ACCESSORIES": "Smart Home & Office Accessories",
    "TRAVEL ACCESSORIES": "Bags, Sleeves & Backpacks",
    "Baseus": "Chargers & Power Banks",
    "Anker": "Chargers & Power Banks",
    "COTEetCI": "Cases & Screen Protectors",

    # Computer Case
    "Computer Accessories > Computer Casing": "Smart Home & Office Accessories",
    "Brands > Components > Computer Case > NZXT": "Smart Home & Office Accessories",
    "Computer Accessories > Computer Accessories": "Smart Home & Office Accessories",

    "Brands > Consumer Notebooks > Core i7 Laptops > Gaming Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops",
    "Mobile Phones > Tecno > Mobile Phones > Tecno": "Mobile Phones",
    "Cameras > Camera Accessories > Camera Accessories": "Camera Accessories (Gimbals, Tripods)",
    "Mobile Accessories > Earphones > WK > Earphones > WK": "Headphones & Earbuds",
    "Smart Watches > Amazfit > Amazfit > Smart Watches": "Smart Watches & Accessories",
    "AUDIO SPEAKERS": "Speakers",
    "Apple Accessories > Lightning Adapters": "Cables & Adapters",
    "Bluetooth Adapters > Computer Accessories": "Cables & Adapters",
    "Mobile Accessories": "Cases & Screen Protectors",
    "Modio > Smart Watches": "Smart Watches & Accessories",
    "Anker > Earphones": "Headphones & Earbuds",
    "Flash Drives & Pen Drives > Kingston": "Storage (SSD, HDD, Pen Drives)",
    "Brands > Computers & Accessories > Monitors > Monitors > Professional Monitors": "Monitors",
    "Laptops > Asus > Asus > Laptops": "Laptops",
    "Mobile Accessories > Earphones": "Headphones & Earbuds",
    "Mobile Phones > Xiaomi > Xiaomi": "Mobile Phones",
    "Brands > Consumer Notebooks > Core i3 Laptops > Laptops > Laptops & Desktops > Laptops & Tabs > Non-Touch Laptops": "Laptops"
}

In [112]:
# assume filtered_df is your DataFrame
filtered_df["category"] = filtered_df["category_path"].map(category_mapping)

# If you want to catch unmapped categories:
filtered_df["category"] = filtered_df["category"].fillna("Other")

In [113]:
filtered_df["category"].value_counts()

,count
category,
Cases & Screen Protectors,5123
Chargers & Power Banks,2412
Cables & Adapters,2382
Mobile Phones,1544
Headphones & Earbuds,1265
Smart Watches & Accessories,1168
Speakers,915
Laptops,699
Smart Home & Office Accessories,597


In [114]:
# print category path value count where category is other
filtered_df[filtered_df["category"] == "Other"]["category_path"].value_counts()

,count
category_path,


In [97]:
# read the json file and crate pd data frame
df = pd.read_csv("/content/dataset.csv")


In [68]:
df.head()

,Unnamed: 0,name,brand,category1,category2
0,0,Himmlisch ST381 Magnetic Sun Shade For Maruti ...,Himmlisch,Home & Kitchen & Automotive,automotive
1,1,Mount Nano MN 110 250 ml Wheel Tire Cleaner,Mount Nano,Home & Kitchen & Automotive,automotive
2,2,Carmity RN-020 Maruti Ciaz Car Grill Cover,Carmity,Home & Kitchen & Automotive,automotive
3,3,Trigcars I20 Car Grill Cover,Trigcars,Home & Kitchen & Automotive,automotive
4,4,Autofurnish Car Cover For Santro Xing,Autofurnish,Home & Kitchen & Automotive,automotive


In [98]:
df["category2"].value_counts()

,count
category2,
Jewellery,1440
Mobiles,1237
Home Furnishing,1236
Women's footware,1200
Men's footware,1200
Laptops,1198
Women's Clothing,1042
automotive,986
Kitchen,616


In [ ]:
# filter only where category2 is Speakers, Camera, Headphones


In [99]:
# filter only where category2 is Speakers, Camera, Headphones
df_filtered_2 = df[df["category2"].isin(["Speakers", "Camera", "Headphones"])].copy()

# Display the filtered DataFrame
display(df_filtered_2.head())

# Display the value counts for the filtered categories
display(df_filtered_2["category2"].value_counts())

,Unnamed: 0,name,brand,category1,category2
986,0,Uniross Compact 9V Battery Charger & 4U AA 100...,Uniross,Electronics,Camera
987,1,SD Extension Arm Rail Camera Mount,SD,Electronics,Camera
988,2,V & B GALLERY Universal 3 in 1 Clip Camera pro...,V & B GALLERY,Electronics,Camera
989,3,SB RETAILS SB02 Flash,SB RETAILS,Electronics,Camera
990,4,Big Mike s 52mm lens hood Lens Hood,Big Mike s,Electronics,Camera


,count
category2,
Speakers,131
Camera,125
Headphones,69


In [101]:
# only name (as product_title) and category 2 as category
df_filtered_2.rename(columns={"name": "product_title", "category2": "category"}, inplace=True)

In [102]:
df_filtered_2 = df_filtered_2[["product_title", "category"]]
df_filtered_2.head()

,product_title,category
986,Uniross Compact 9V Battery Charger & 4U AA 100...,Camera
987,SD Extension Arm Rail Camera Mount,Camera
988,V & B GALLERY Universal 3 in 1 Clip Camera pro...,Camera
989,SB RETAILS SB02 Flash,Camera
990,Big Mike s 52mm lens hood Lens Hood,Camera


In [103]:
df = pd.read_csv("/content/pricerunner_aggregate.csv")
df.head()

,Product ID,Product Title,Merchant ID,Cluster ID,Cluster Label,Category ID,Category Label
0,1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
1,2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,4,apple iphone 8 plus 64gb space grey,4,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones


In [84]:
# Category Label value count
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Product ID       35311 non-null  int64 
 1   Product Title    35311 non-null  object
 2    Merchant ID     35311 non-null  int64 
 3    Cluster ID      35311 non-null  int64 
 4    Cluster Label   35311 non-null  object
 5    Category ID     35311 non-null  int64 
 6    Category Label  35311 non-null  object
dtypes: int64(4), object(3)
memory usage: 1.9+ MB


In [85]:
df[" Category Label"].value_counts()

,count
Category Label,
Fridge Freezers,5501
Mobile Phones,4081
Washing Machines,4044
CPUs,3862
Fridges,3584
TVs,3564
Dishwashers,3424
Digital Cameras,2697
Microwaves,2342


In [104]:
# df[df[" Category Label"] == "CPUs"] where have fifferent " Cluster ID" get 1 st one from different clusters
filtered_df_3 = df[df[" Category Label"] == "CPUs"]

# Drop duplicates based on 'Cluster ID' to get the first occurrence per cluster
first_per_cluster = filtered_df.drop_duplicates(subset=" Cluster ID", keep="first")


In [105]:
first_per_cluster = first_per_cluster.head(200)

In [106]:
first_per_cluster[["Product Title", " Category Label"]]

,Product Title,Category Label
7645,amd ryzen 7 eight core 1700x 3.80ghz socket am...,CPUs
7654,intel core intel core i7 7700 processor 8m cac...,CPUs
7667,intel core intel core i7 8700k processor 12m c...,CPUs
7678,intel core intel core i3 8100 processor 6m cac...,CPUs
7689,intel core intel core i7 7700k processor 8m ca...,CPUs
...,...,...
8599,intel pentium g4500t lga11513mb cache tray,CPUs
8602,intel pentium dual core g4520 3.60ghz s1151 3m...,CPUs
8605,intel xeon e5 2618lv3 2.3 ghz 8 core 16 thread...,CPUs
8608,intel xeon e5 2630v3 processor 2.4 ghz box 20 ...,CPUs


In [107]:

first_per_cluster.rename(columns={"Product Title": "product_title", " Category Label": "category"}, inplace=True)

In [115]:
filtered_df.head()

,shop_product_id,shop_name,product_title,category_path,category
0,a1059bb64dd95625fc570f12d2b51faf,lifemobile.lk,Huawei Watch GT 4 46MM – Black,Smart Watches > Huawei > Huawei > Smart Watches,Smart Watches & Accessories
2,57b22d9f4b9ea53e8b9a52b4c9320620,lifemobile.lk,Xiaomi Mi Band 8,Fitness Trackers > Xiaomi > Fitness Trackers >...,Smart Watches & Accessories
3,a2d29845d0b9b6f900559602c2899530,lifemobile.lk,TP-Link M7200 4G LTE Mobile Wi-Fi,Computer Peripherals > WiFi Routers > WiFi Rou...,Networking (Routers)
4,d4321f6a2275185a3fb5440685265cdf,lifemobile.lk,Apple MGN63 13.3-inch MacBook Air M1 Chip with...,Laptops > Apple MacBook > Laptops > Apple MacBook,Laptops
5,1d128edfdee679016c9dcc7d8a4addef,lifemobile.lk,JBL Flip 6,Bluetooth Speakers > JBL > Bluetooth Speakers ...,Speakers


In [116]:
filtered_df= filtered_df[["product_title", "category"]]

In [118]:
filtered_df.shape, first_per_cluster.shape, df_filtered_2.shape

((20915, 2), (200, 7), (325, 2))

In [119]:
# combined all 3 filtered_df, first_per_cluster, df_filtered_2
df = pd.concat([filtered_df, first_per_cluster, df_filtered_2])

In [120]:
df.shape

(21440, 7)

In [121]:
df["category"].value_counts()

,count
category,
Cases & Screen Protectors,5123
Chargers & Power Banks,2412
Cables & Adapters,2382
Mobile Phones,1544
Headphones & Earbuds,1265
Smart Watches & Accessories,1168
Speakers,1046
Laptops,699
Smart Home & Office Accessories,597


In [122]:
# rename the categories in "category" cloumn rename "Headphones" -> "Headphones & Earbuds" , ""Camera"" -> "Cameras & Drones", "CPUs (Processors)" - "CPUs"

df["category"] = df["category"].replace({
    "Headphones": "Headphones & Earbuds",
    "Camera": "Cameras & Drones",
    "CPUs (Processors)": "CPUs",
    "Storage (SSD, HDD, Pen Drives)":"Storage",
    "Camera Accessories (Gimbals, Tripods)": "Camera Accessories",
    "Networking (Routers)":"Networking",
    "Gaming Peripherals (Chairs, etc.)":"Gaming Peripherals",
    "RAM (Memory)":"Memory",
    "Graphic Cards (VGA)": "Graphic Cards"

})

In [125]:
df["category"].value_counts()

,count
category,
Cases & Screen Protectors,5123
Chargers & Power Banks,2412
Cables & Adapters,2382
Mobile Phones,1544
Headphones & Earbuds,1334
Smart Watches & Accessories,1168
Speakers,1046
Laptops,699
Smart Home & Office Accessories,597


In [127]:
df = df[["product_title", "category"]]

In [128]:
df.head()

,product_title,category
0,Huawei Watch GT 4 46MM – Black,Smart Watches & Accessories
2,Xiaomi Mi Band 8,Smart Watches & Accessories
3,TP-Link M7200 4G LTE Mobile Wi-Fi,Networking
4,Apple MGN63 13.3-inch MacBook Air M1 Chip with...,Laptops
5,JBL Flip 6,Speakers


In [129]:
df.shape

(21440, 2)

In [130]:
df["category"].value_counts()

,count
category,
Cases & Screen Protectors,5123
Chargers & Power Banks,2412
Cables & Adapters,2382
Mobile Phones,1544
Headphones & Earbuds,1334
Smart Watches & Accessories,1168
Speakers,1046
Laptops,699
Smart Home & Office Accessories,597


In [131]:
# save df data frame
df.to_csv("output/products_summary_filtered.csv", index=False, encoding="utf-8")